## Asignamos la extencion del archivo

In [ ]:
import os

def cambiar_extension(carpeta, nueva_extension, modo="ignorar"):
    """
    Cambia la extensión de los archivos de una carpeta.

    modos:
    - 'ignorar'       → no renombra si el archivo ya existe
    - 'sobrescribir'  → reemplaza el archivo existente
    - 'sufijo'        → agrega _1, _2, etc.
    """

    if not nueva_extension.startswith("."):
        nueva_extension = "." + nueva_extension

    for archivo in os.listdir(carpeta):
        ruta_original = os.path.join(carpeta, archivo)

        if not os.path.isfile(ruta_original):
            continue

        nombre, ext = os.path.splitext(archivo)

        # Si ya tiene la extensión deseada, se ignora
        if ext.lower() == nueva_extension.lower():
            continue

        nueva_ruta = os.path.join(carpeta, nombre + nueva_extension)

        # === CASO 1: IGNORAR ===
        if modo == "ignorar":
            if os.path.exists(nueva_ruta):
                print(f"⚠️ Ignorado: {nueva_ruta}")
                continue

        # === CASO 2: SOBRESCRIBIR ===
        elif modo == "sobrescribir":
            if os.path.exists(nueva_ruta):
                os.remove(nueva_ruta)

        # === CASO 3: SUFIJO ===
        elif modo == "sufijo":
            contador = 1
            base = nueva_ruta
            while os.path.exists(nueva_ruta):
                nueva_ruta = os.path.join(
                    carpeta, f"{nombre}_{contador}{nueva_extension}"
                )
                contador += 1

        else:
            raise ValueError("Modo no válido: usar 'ignorar', 'sobrescribir' o 'sufijo'")

        os.rename(ruta_original, nueva_ruta)


cambiar_extension("Temblores_1964-1999", ".txt", modo="sufijo")


In [4]:
import os

def cambiar_extension(carpeta, nueva_extension):
    """
    Cambia la extensión de todos los archivos de una carpeta.

    Parámetros:
    carpeta (str): Ruta de la carpeta
    nueva_extension (str): Nueva extensión (ej. '.txt' o '.csv')
    """

    if not nueva_extension.startswith("."):
        nueva_extension = "." + nueva_extension

    for archivo in os.listdir(carpeta):
        ruta_original = os.path.join(carpeta, archivo)

        if os.path.isfile(ruta_original):
            nombre, _ = os.path.splitext(archivo)
            nueva_ruta = os.path.join(carpeta, nombre + nueva_extension)
            os.rename(ruta_original, nueva_ruta)

    print(f"Extensión cambiada a {nueva_extension} para los archivos en {carpeta}")


## Ejemplo de uso
cambiar_extension("Temblores_2000-2020", ".txt")


Extensión cambiada a .txt para los archivos en Temblores_2000-2020


In [52]:
import os

def cambiar_extensiones(directorio, ext_origen=".txt", ext_destino=".csv"):
    """
    Cambia la extensión de todos los archivos en un directorio.
    
    Args:
        directorio (str): Ruta de la carpeta donde están los archivos.
        ext_origen (str): Extensión actual de los archivos (ejemplo: ".txt").
        ext_destino (str): Nueva extensión deseada (ejemplo: ".csv").
    """
    for archivo in os.listdir(directorio):
        if archivo.endswith(ext_origen):
            ruta_vieja = os.path.join(directorio, archivo)
            nuevo_nombre = archivo.replace(ext_origen, ext_destino)
            ruta_nueva = os.path.join(directorio, nuevo_nombre)
            os.rename(ruta_vieja, ruta_nueva)
            print(f"Renombrado: {archivo} → {nuevo_nombre}")

# Ejemplo de uso
cambiar_extensiones("2017/2017-01-12_102658")


Renombrado: AL0120170112102658.txt → AL0120170112102658.csv
Renombrado: BA4920170112102658.txt → BA4920170112102658.csv
Renombrado: BO3920170112102658.txt → BO3920170112102658.csv
Renombrado: CI0520170112102658.txt → CI0520170112102658.csv
Renombrado: CJ0320170112102658.txt → CJ0320170112102658.csv
Renombrado: DX3720170112102658.txt → DX3720170112102658.csv
Renombrado: GR2720170112102658.txt → GR2720170112102658.csv
Renombrado: LI3320170112102658.txt → LI3320170112102658.csv
Renombrado: LV1720170112102658.txt → LV1720170112102658.csv
Renombrado: NZ3120170112102658.txt → NZ3120170112102658.csv
Renombrado: S16020170112102658.txt → S16020170112102658.csv
Renombrado: S26020170112102658.txt → S26020170112102658.csv
Renombrado: S36020170112102658.txt → S36020170112102658.csv
Renombrado: SP5120170112102658.txt → SP5120170112102658.csv
Renombrado: SS6020170112102658.txt → SS6020170112102658.csv
Renombrado: TH3520170112102658.txt → TH3520170112102658.csv
Renombrado: TP1320170112102658.txt → TP1

## Realizamos el catalogo de los registros de los sismos

In [55]:
import os
import pandas as pd
from datetime import datetime
import re

def construir_catalogo(directorio):
    registros = []

    for archivo in os.listdir(directorio):
        if archivo.endswith(".txt") or archivo.endswith(".csv"):
            ruta = os.path.join(directorio, archivo)

            with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
                contenido = f.read()

            # -------- FECHA --------
            fecha_match = re.search(r"FECHA DEL SISMO \(GMT\)\s*:\s*(\d{2})/([A-Z]{3})/(\d{2})", contenido)
            if fecha_match:
                dia, mes_txt, anio = fecha_match.groups()
                meses = {
                    "ENE": "01", "FEB": "02", "MAR": "03", "ABR": "04",
                    "MAY": "05", "JUN": "06", "JUL": "07", "AGO": "08",
                    "SEP": "09", "OCT": "10", "NOV": "11", "DIC": "12"
                }
                mes = meses.get(mes_txt, "01")
                fecha = f"20{anio}-{mes}-{dia}"
            else:
                continue  # si no hay fecha, no sirve el archivo

            # -------- HORA --------
            hora_match = re.search(r"HORA EPICENTRO \(GMT\)\s*:\s*([\d:.]+)", contenido)
            if not hora_match:
                continue
            hora = hora_match.group(1).split(".")[0]

            # -------- MAGNITUD --------
            mag_match = re.search(r"Mc=\s*([\d.]+)", contenido)
            magnitud = float(mag_match.group(1)) if mag_match else None

            # -------- LATITUD --------
            lat_match = re.search(r"COORDENADAS DEL EPICENTRO\s*:\s*([\d.]+)\s*LAT", contenido)
            latitud = float(lat_match.group(1)) if lat_match else None

            # -------- LONGITUD --------
            lon_match = re.search(r":\s*([\d.]+)\s*LONG\.?\s*W", contenido)
            longitud = -float(lon_match.group(1)) if lon_match else None

            # -------- PROFUNDIDAD --------
            prof_match = re.search(r"PROFUNDIDAD FOCAL \(km\)\s*:\s*([\d.]+)", contenido)
            profundidad = float(prof_match.group(1)) if prof_match else None

            # -------- DATETIME --------
            fecha_hora = datetime.strptime(f"{fecha} {hora}", "%Y-%m-%d %H:%M:%S")

            registros.append({
                "datetime": fecha_hora,
                "magnitude": magnitud,
                "latitude": latitud,
                "longitude": longitud,
                "depth": profundidad
            })

    return pd.DataFrame(registros)

# Ejemplo de uso
catalogo = construir_catalogo("2017/2017-01-12_102658")
print(catalogo.head(20))
print(f"\nTotal registros: {len(catalogo)}")
print("-------------------------------------------------------------\n")

df = construir_catalogo("2017/2017-01-12_102658")

# df.to_csv("catalogo_sismos.csv", index=False)
# print("Catálogo sísmico generado correctamente")


              datetime  magnitude  latitude  longitude  depth
0  2017-01-12 10:26:58        5.0     16.59   -99.1453   39.0
1  2017-01-12 10:26:58        5.0     16.59   -99.1450   39.0
2  2017-01-12 10:26:58        5.0     16.59   -99.1047   39.0
3  2017-01-12 10:26:58        5.0     16.59   -99.1653   39.0
4  2017-01-12 10:26:58        5.0     16.59   -99.1567   39.0
5  2017-01-12 10:26:58        5.0     16.59   -99.1439   39.0
6  2017-01-12 10:26:58        5.0     16.59   -99.1797   39.0
7  2017-01-12 10:26:58        5.0     16.59   -98.9631   39.0
8  2017-01-12 10:26:58        5.0     16.59   -99.1275   39.0
9  2017-01-12 10:26:58        5.0     16.59   -99.0247   39.0
10 2017-01-12 10:26:58        5.0     16.59   -99.1470   39.0
11 2017-01-12 10:26:58        5.0     16.59   -99.1470   39.0
12 2017-01-12 10:26:58        5.0     16.59   -99.1470   39.0
13 2017-01-12 10:26:58        5.0     16.59   -99.1189   39.0
14 2017-01-12 10:26:58        5.0     16.59   -99.1470   39.0
15 2017-

## Cargamos todo el contenido de la carpeta "2017"

In [59]:
import os
import re
import pandas as pd
from datetime import datetime

# ================================
# CONFIGURACIÓN
# ================================

CARPETA_RAIZ = "2017"   # carpeta donde están todas las subcarpetas de sismos
SALIDA = "catalogo_sismos.csv"

# ================================
# MESES EN ESPAÑOL → INGLÉS
# ================================

MESES = {
    "ENE": "Jan",
    "FEB": "Feb",
    "MAR": "Mar",
    "ABR": "Apr",
    "MAY": "May",
    "JUN": "Jun",
    "JUL": "Jul",
    "AGO": "Aug",
    "SEP": "Sep",
    "OCT": "Oct",
    "NOV": "Nov",
    "DIC": "Dec"
}

def convertir_fecha_espanol(fecha_str):
    for mes_es, mes_en in MESES.items():
        if mes_es in fecha_str:
            fecha_str = fecha_str.replace(mes_es, mes_en)
            break
    return fecha_str

# ================================
# FUNCIONES DE EXTRACCIÓN
# ================================

def extraer_campo(texto, patron):
    match = re.search(patron, texto)
    return match.group(1).strip() if match else None


def extraer_datos_sismo_archivo(ruta_archivo):
    with open(ruta_archivo, "r", encoding="latin-1", errors="ignore") as f:
        contenido = f.read()

    # Fecha y hora
    fecha_str = extraer_campo(contenido, r"FECHA DEL SISMO .*?:\s*([0-9\/A-Z]+)")
    hora_str  = extraer_campo(contenido, r"HORA EPICENTRO .*?:\s*([0-9:\.]+)")

    # Magnitud
    magnitud = extraer_campo(contenido, r"Mc=([0-9\.]+)")

    # Latitud y Longitud
    lat = extraer_campo(contenido, r"([0-9\.]+)\s+LAT")
    lon = extraer_campo(contenido, r"([0-9\.]+)\s+LONG")

    # Profundidad
    profundidad = extraer_campo(contenido, r"PROFUNDIDAD FOCAL .*?:\s*([0-9\.]+)")

    # ================================
    # NORMALIZACIÓN CORRECTA DE FECHA
    # ================================

    try:
        fecha_str = convertir_fecha_espanol(fecha_str)
        hora_limpia = hora_str.split(".")[0]
        fecha = datetime.strptime(fecha_str + " " + hora_limpia, "%d/%b/%y %H:%M:%S")
    except:
        fecha = None

    return {
        "datetime": fecha,
        "magnitude": float(magnitud) if magnitud else None,
        "latitude": float(lat) if lat else None,
        "longitude": -float(lon) if lon else None,  # Oeste negativo
        "depth": float(profundidad) if profundidad else None
    }

# ================================
# RECOLECTOR MAESTRO DE SISMOS
# ================================

catalogo = []

for carpeta_sismo in sorted(os.listdir(CARPETA_RAIZ)):
    ruta_sismo = os.path.join(CARPETA_RAIZ, carpeta_sismo)

    if os.path.isdir(ruta_sismo):
        archivos = os.listdir(ruta_sismo)

        if len(archivos) == 0:
            continue

        # Tomamos SOLO EL PRIMER ARCHIVO como representante del evento
        archivo_representativo = os.path.join(ruta_sismo, archivos[0])

        datos = extraer_datos_sismo_archivo(archivo_representativo)

        print(f"Procesado: {carpeta_sismo} → {datos}")
        catalogo.append(datos)

# ================================
# CONSTRUCCIÓN DEL CSV FINAL
# ================================

df = pd.DataFrame(catalogo)
df = df.dropna()
df = df.sort_values("datetime")
df.to_csv(SALIDA, index=False)

print("\n Catálogo histórico generado correctamente:")
print(df)


Procesado: 2017-01-12_102658 → {'datetime': datetime.datetime(2017, 1, 12, 10, 26, 58), 'magnitude': 5.0, 'latitude': 19.4356, 'longitude': -99.1453, 'depth': 39.0}
Procesado: 2017-02-02_005209 → {'datetime': datetime.datetime(2017, 2, 2, 0, 52, 9), 'magnitude': 3.7, 'latitude': 19.4356, 'longitude': -99.1453, 'depth': 8.0}
Procesado: 2017-02-13_072930 → {'datetime': datetime.datetime(2017, 2, 13, 7, 29, 30), 'magnitude': 5.0, 'latitude': 19.4356, 'longitude': -99.1453, 'depth': 8.0}
Procesado: 2017-08-18_051512 → {'datetime': datetime.datetime(2017, 8, 18, 5, 15, 12), 'magnitude': 5.3, 'latitude': 19.4097, 'longitude': -99.145, 'depth': 5.0}
Procesado: 2017-09-08_044917 → {'datetime': datetime.datetime(2017, 9, 8, 4, 49, 17), 'magnitude': 8.2, 'latitude': 19.429, 'longitude': -99.0584, 'depth': 45.0}
Procesado: 2017-09-19_181439 → {'datetime': datetime.datetime(2017, 9, 19, 18, 14, 39), 'magnitude': 7.1, 'latitude': 19.429, 'longitude': -99.0584, 'depth': 51.0}
Procesado: 2017-09-23_1